<a href="https://colab.research.google.com/github/rhodes-byu/stat-486/blob/main/notebooks/06a-imbalance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b>Student Activity Notebook</b></p>

# Class Imbalance in Machine Learning: Student Activity

## Learning Objectives

By the end of this notebook, you will be able to:
- Recognize class imbalance in datasets and understand its impact on model performance
- Build and evaluate classification pipelines on imbalanced data
- Implement resampling techniques (undersampling and oversampling) using pipelines
- Apply class weighting to handle imbalance
- Evaluate models using appropriate metrics (precision, recall, F1, ROC-AUC, PR-AUC)
- Compare different approaches and select the best strategy for a given problem

## The Problem

**Class imbalance** occurs when the distribution of target classes in a dataset is significantly skewed. This is common in real-world scenarios:
- **Fraud detection**: Fraudulent transactions are rare (~0.1-2% of all transactions)
- **Medical diagnosis**: Diseased cases are much less frequent than healthy ones
- **Customer churn**: Most customers stay active
- **Anomaly detection**: Anomalies are rare by definition

### Why It Matters

Models trained on imbalanced data tend to:
- **Favor the majority class** to maximize overall accuracy
- **Perform poorly on the minority class** (which is usually what we care about most)
- **Achieve misleadingly high accuracy** by simply predicting the majority class

**Example**: A model that always predicts "not fraud" on a 99:1 imbalanced dataset achieves 99% accuracy but detects zero fraud cases!

## Your Task

In this notebook, you'll work with an imbalanced dataset and:
1. Build a baseline classification pipeline
2. Implement different strategies to handle imbalance
3. Compare approaches using appropriate evaluation metrics
4. Determine the best solution for this problem

## Section 1: Import Libraries and Generate Data

First, let's import the necessary libraries and create a synthetic imbalanced dataset.

**Note**: Data generation code is provided since creating realistic imbalanced datasets requires specific parameters.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score, roc_curve
)

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")

### Generate Synthetic Imbalanced Dataset

**This code is provided** to create a realistic imbalanced classification problem with:
- 5,000 samples total
- 25 features (15 informative, 5 redundant, 5 noise)
- Severe imbalance: 95% majority class, 5% minority class
- Some label noise to increase difficulty

In [ ]:
from sklearn.datasets import make_classification

# Create synthetic imbalanced dataset
X, y = make_classification(
    n_samples=5000,
    n_features=25,
    n_informative=15,
    n_redundant=5,
    weights=[0.95, 0.05],  # 95% class 0, 5% class 1
    flip_y=0.05,           # 5% label noise
    random_state=42
)

# Create DataFrame for inspection
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
df['target'] = y

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

## Section 2: Explore the Class Distribution

**YOUR TASK**: Analyze the class distribution in the dataset.

Complete the code below to:
1. Calculate the count and percentage for each class
2. Compute the imbalance ratio (majority:minority)
3. Create a visualization of the class distribution

In [ ]:
# TODO: Calculate class distribution
# Hint: Use np.unique(y, return_counts=True) or pd.Series(y).value_counts()

class_0_count = # YOUR CODE HERE
class_1_count = # YOUR CODE HERE

class_0_pct = # YOUR CODE HERE (as percentage)
class_1_pct = # YOUR CODE HERE (as percentage)

imbalance_ratio = # YOUR CODE HERE (majority / minority)

print("Class Distribution:")
print(f"Class 0 (Majority): {class_0_count} samples ({class_0_pct:.1f}%)")
print(f"Class 1 (Minority): {class_1_count} samples ({class_1_pct:.1f}%)")
print(f"\nImbalance ratio: {imbalance_ratio:.1f}:1")

In [ ]:
# TODO: Create a bar plot visualizing class distribution
# Hint: Use matplotlib or seaborn
# Consider using a log scale for the y-axis due to the large difference

fig, ax = plt.subplots(figsize=(8, 5))

# YOUR CODE HERE - create bar plot
# Suggested: ax.bar(...) with class names and counts
# Consider: ax.set_yscale('log') for better visualization

plt.tight_layout()
plt.show()

## Section 3: Split Data and Build Baseline Pipeline

**YOUR TASK**: Create a train-test split and build a baseline classification pipeline.

**Important**: When splitting imbalanced data, use `stratify=y` to maintain class proportions in both sets.

In [ ]:
# TODO: Split the data into training and testing sets (80/20 split)
# Use stratified sampling to preserve class distribution
# Set random_state=42 for reproducibility

X_train, X_test, y_train, y_test = # YOUR CODE HERE

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set class distribution:")
print(f"  Class 0: {(y_train == 0).sum()} ({(y_train == 0).sum() / len(y_train) * 100:.1f}%)")
print(f"  Class 1: {(y_train == 1).sum()} ({(y_train == 1).sum() / len(y_train) * 100:.1f}%)")

### Build a Baseline Classification Pipeline

**YOUR TASK**: Create a scikit-learn pipeline with:
1. StandardScaler for feature scaling
2. LogisticRegression for classification (use max_iter=1000, random_state=42)

Then train it on the training data.

In [ ]:
# TODO: Create a pipeline with StandardScaler and LogisticRegression
# Hint: Use Pipeline from sklearn.pipeline

baseline_pipeline = Pipeline([
    # YOUR CODE HERE - add (name, transformer/estimator) tuples
])

# TODO: Fit the pipeline on training data
# YOUR CODE HERE

print("✓ Baseline pipeline trained")

## Section 4: Evaluation Function for Imbalanced Data

**This function is provided** because comprehensive evaluation requires many metrics.

Key metrics for imbalanced classification:
- **Precision**: Of predicted positives, how many are correct?
- **Recall (Sensitivity)**: Of actual positives, how many did we find?
- **F1-Score**: Harmonic mean of precision and recall
- **Specificity**: Of actual negatives, how many did we correctly identify?
- **ROC-AUC**: Area under ROC curve (threshold-independent)
- **PR-AUC**: Precision-Recall AUC (focuses on minority class)

**Note**: Never rely solely on accuracy for imbalanced data!

In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    """
    Comprehensive evaluation function for imbalanced classification.

    Parameters:
    -----------
    model : fitted pipeline or estimator
        Must have predict() and predict_proba() methods
    X_test : array-like
        Test features
    y_test : array-like
        True test labels
    model_name : str
        Name to display in output

    Returns:
    --------
    dict : Dictionary of metric values
    """
    # Get predictions and probabilities
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    # Print results
    print(f"\n{'=' * 70}")
    print(f"{model_name}")
    print(f"{'=' * 70}")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}  (of predicted +, how many correct)")
    print(f"Recall:         {recall:.4f}  (of actual +, how many found)")
    print(f"Specificity:    {specificity:.4f}  (of actual -, how many correct)")
    print(f"F1-Score:       {f1:.4f}  (balance precision & recall)")
    print(f"ROC-AUC:        {roc_auc:.4f}  (threshold-independent)")
    print(f"PR-AUC:         {pr_auc:.4f}  (minority class focus)")

    print(f"\nConfusion Matrix:")
    print(f"                Predicted Neg | Predicted Pos")
    print(f"Actual Negative |    {tn:5d}     |    {fp:5d}")
    print(f"Actual Positive |    {fn:5d}     |    {tp:5d}")

    return {
        'accuracy': accuracy, 'precision': precision, 'recall': recall,
        'f1': f1, 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'specificity': specificity
    }

### Evaluate Your Baseline Model

**YOUR TASK**: Use the provided evaluation function to assess your baseline pipeline's performance.

In [ ]:
# TODO: Evaluate the baseline pipeline using the evaluate_model function
baseline_metrics = # YOUR CODE HERE

# Reflect on the results:
# - What is the accuracy? Is it high or low?
# - What is the recall? Are we catching most minority class instances?
# - How does the model perform on the minority class overall?

## Section 5: Understanding the Naive Baseline

Before implementing solutions, let's understand the problem better.

**YOUR TASK**: Create a "naive" classifier that always predicts the majority class (Class 0). Calculate its accuracy to understand why accuracy is misleading for imbalanced data.

In [ ]:
# TODO: Create predictions that always predict class 0
# Hint: np.zeros_like(y_test)
y_naive = # YOUR CODE HERE

# TODO: Calculate accuracy for this naive classifier
naive_accuracy = # YOUR CODE HERE

print(f"Naive Classifier (Always Predict Class 0):")
print(f"Accuracy: {naive_accuracy:.4f}")
print(f"\nThis {naive_accuracy:.1%} accuracy is USELESS!")
print(f"It never detects the minority class (Class 1).")

# TODO: Print confusion matrix for naive classifier
print(f"\nConfusion Matrix:")
# YOUR CODE HERE

## Section 6: Undersampling Approach

**Undersampling** reduces the majority class to balance the dataset.

**Strategy provided**: We'll use `RandomUnderSampler` from the `imblearn` library.

**YOUR TASK**: Build a pipeline that includes undersampling, scaling, and classification.

**Note**: sklearn's `Pipeline` works seamlessly with imblearn resampling steps (it automatically detects and calls `fit_resample` when needed).

In [ ]:
# Example: How RandomUnderSampler works (PROVIDED)
print("Example: RandomUnderSampler demonstration")
print(f"Original training data: Class 0 = {(y_train == 0).sum()}, Class 1 = {(y_train == 1).sum()}")

# Apply undersampling
undersampler = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = undersampler.fit_resample(X_train, y_train)

print(f"After undersampling: Class 0 = {(y_train_under == 0).sum()}, Class 1 = {(y_train_under == 1).sum()}")
print(f"\n✓ Dataset balanced by removing majority class samples")
print(f"  Trade-off: Discards potentially useful information")

In [2]:
# TODO: Create a pipeline with RandomUnderSampler, StandardScaler, and LogisticRegression
# Use imblearn's Pipeline - it works with imblearn samplers!
from imblearn.pipeline import Pipeline

undersample_pipeline = Pipeline([
    # YOUR CODE HERE - add (name, transformer/estimator) tuples
    # Order: undersampler -> scaler -> classifier
])

# TODO: Fit the pipeline on training data
# YOUR CODE HERE

print("✓ Undersampling pipeline trained")

✓ Undersampling pipeline trained


In [ ]:
# TODO: Evaluate the undersampling pipeline
undersample_metrics = # YOUR CODE HERE

# Reflect:
# - How did recall change compared to baseline?
# - What happened to precision?
# - Is the F1-score better or worse?

## Section 7: Oversampling with SMOTE

**SMOTE (Synthetic Minority Over-sampling Technique)** generates synthetic minority class samples by interpolating between existing samples.

**Strategy provided**: SMOTE creates new samples rather than duplicating existing ones, reducing overfitting risk.

**YOUR TASK**: Build a pipeline using SMOTE for oversampling.

In [ ]:
# Example: How SMOTE works (PROVIDED)
print("Example: SMOTE demonstration")
print(f"Original training data: Class 0 = {(y_train == 0).sum()}, Class 1 = {(y_train == 1).sum()}")

# Apply SMOTE
smote = SMOTE(k_neighbors=5, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"After SMOTE: Class 0 = {(y_train_smote == 0).sum()}, Class 1 = {(y_train_smote == 1).sum()}")
print(f"\n✓ Synthetic samples created by interpolating between minority class neighbors")
print(f"  Advantage: No information loss, diverse synthetic samples")

In [ ]:
# TODO: Create a pipeline with SMOTE, StandardScaler, and LogisticRegression
# Use sklearn's Pipeline - it automatically handles SMOTE's fit_resample method
# Use k_neighbors=5 for SMOTE

smote_pipeline = Pipeline([
    # YOUR CODE HERE - add (name, transformer/estimator) tuples
])

# TODO: Fit the pipeline on training data
# YOUR CODE HERE

print("✓ SMOTE pipeline trained")

In [ ]:
# TODO: Evaluate the SMOTE pipeline
smote_metrics = # YOUR CODE HERE

# Reflect:
# - How does SMOTE compare to undersampling?
# - Which metrics improved?
# - Is this better than the baseline?

## Section 8: Class Weighting Approach

**Class weighting** assigns higher penalties to minority class misclassifications during training.

**Advantage**: No data manipulation needed—works directly on original imbalanced data!

**YOUR TASK**: Build a pipeline using class weighting in LogisticRegression.

In [ ]:
# TODO: Create a pipeline with StandardScaler and LogisticRegression
# Set class_weight='balanced' in LogisticRegression to automatically handle imbalance
# Use max_iter=1000, random_state=42

weighted_pipeline = Pipeline([
    # YOUR CODE HERE
])

# TODO: Fit the pipeline on training data
# YOUR CODE HERE

print("✓ Class-weighted pipeline trained")
print("\n  Note: class_weight='balanced' automatically calculates weights")
print("  inversely proportional to class frequencies")

In [ ]:
# TODO: Evaluate the class-weighted pipeline
weighted_metrics = # YOUR CODE HERE

# Reflect:
# - How does class weighting compare to resampling techniques?
# - What are the advantages of this approach?

## Section 9: Comprehensive Comparison

**YOUR TASK**: Compare all your approaches to determine which performs best.

Create a comparison table showing all metrics for each approach.

In [ ]:
# TODO: Create a dictionary of all model metrics
# Format: {'Model Name': metrics_dict}

all_models = {
    # YOUR CODE HERE - add all four approaches
    # 'Baseline': baseline_metrics,
    # etc.
}

# TODO: Create a pandas DataFrame from the dictionary and display it
comparison_df = # YOUR CODE HERE

# Select relevant metrics for comparison
comparison_df = comparison_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']]

print("\n" + "="*80)
print("MODEL COMPARISON: ALL APPROACHES")
print("="*80)
print(comparison_df.round(4))

### Visualize the Comparison

**YOUR TASK**: Create visualizations to compare model performance across different metrics.

In [ ]:
# TODO: Create bar plots comparing all models across key metrics
# Suggested approach: Create subplots for different metrics

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Comparison of Imbalance Handling Approaches', fontsize=16, fontweight='bold')

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
axes_flat = axes.flatten()

# TODO: For each metric, create a horizontal bar plot
# Hint: Use comparison_df[metric] to get values for each model
# Use axes_flat[idx] to access each subplot

for idx, metric in enumerate(metrics_to_plot):
    ax = axes_flat[idx]
    # YOUR CODE HERE
    # Suggested: ax.barh() for horizontal bars
    # Add metric name as xlabel
    # Consider coloring bars based on performance (optional)

plt.tight_layout()
plt.show()

## Section 10: ROC and Precision-Recall Curves

**This section provides** code to generate ROC and PR curves, as these require multiple model probability predictions.

Understanding these curves:
- **ROC Curve**: Shows trade-off between True Positive Rate (Recall) and False Positive Rate
- **PR Curve**: Focuses on minority class performance—more informative for imbalanced data

In [ ]:
# Precision-Recall Curves (PROVIDED)
fig, ax = plt.subplots(figsize=(10, 7))

models_dict = {
    'Baseline': (baseline_pipeline, 'gray'),
    'Undersampling': (undersample_pipeline, 'blue'),
    'SMOTE': (smote_pipeline, 'red'),
    'Class Weights': (weighted_pipeline, 'green')
}

for name, (model, color) in models_dict.items():
    y_scores = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_scores)
    pr_auc = average_precision_score(y_test, y_scores)

    ax.plot(recall, precision, marker='o', label=f'{name} (AP={pr_auc:.3f})',
            color=color, linewidth=2, markersize=4)

ax.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves: Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Higher and more right-ward curves indicate better minority class detection")

In [ ]:
# ROC Curves (PROVIDED)
fig, ax = plt.subplots(figsize=(10, 7))

for name, (model, color) in models_dict.items():
    y_scores = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = roc_auc_score(y_test, y_scores)

    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})',
            color=color, linewidth=2, marker='o', markersize=4)

# Plot diagonal (random classifier)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC=0.500)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves: Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Higher curves (toward top-left) indicate better overall performance")

## Section 11: Analysis and Conclusions

**YOUR TASK**: Based on your results, answer the following questions.

### Questions to Consider:

1. **Which approach achieved the highest recall?** Why is recall important for this problem?

2. **Which approach achieved the best F1-score?** What does this tell us about the balance between precision and recall?

3. **How did the baseline model perform?** Why was its performance problematic despite high accuracy?

4. **Compare undersampling vs. SMOTE**: Which performed better? What are the trade-offs?

5. **What about class weighting?** How does it compare to resampling techniques? What are its advantages?

6. **Which approach would you recommend for this problem?** Justify your answer based on the metrics and practical considerations.

7. **When would you choose each approach?**
   - Undersampling: _______________
   - SMOTE: _______________
   - Class Weighting: _______________

# YOUR ANALYSIS HERE

Write your responses to the questions above in this markdown cell.

### 1. Highest Recall
[Your answer here]

### 2. Best F1-Score
[Your answer here]

### 3. Baseline Performance
[Your answer here]

### 4. Undersampling vs. SMOTE
[Your answer here]

### 5. Class Weighting
[Your answer here]

### 6. Recommendation
[Your answer here]

### 7. When to Use Each Approach
[Your answer here]

## Key Takeaways

### When to Use Each Technique:

| Approach | Best For | Pros | Cons |
|----------|----------|------|------|
| **Undersampling** | Large datasets, extreme imbalance | Fast, reduces training time | Loses majority class information |
| **SMOTE** | Most real-world problems | Creates diverse synthetic samples | Can generate noisy samples |
| **Class Weighting** | Quick baseline, production models | No data manipulation, simple | May need parameter tuning |

### Critical Metrics for Imbalanced Data:

❌ **NEVER rely solely on accuracy!**

✅ **Always evaluate using:**
- **Recall**: Percentage of minority cases detected
- **Precision**: Percentage of positive predictions that are correct
- **F1-Score**: Balance between precision and recall
- **PR-AUC**: Best overall metric for imbalanced data (focuses on minority class)

### Decision Guidelines:

```
Start with → Class Weighting (fastest, no data manipulation)
            ↓
Still poor performance? → Try SMOTE (generally best results)
            ↓
Dataset too large? → Try Undersampling (faster training)
            ↓
Need more control? → Tune class weights or resampling ratios
```

### Best Practices:

1. ✓ **Always** check class distribution before modeling
2. ✓ **Use stratified splitting** to maintain class proportions
3. ✓ **Evaluate with multiple metrics** (never just accuracy)
4. ✓ **Start simple** (class weighting) before complex approaches
5. ✓ **Consider business costs** of false positives vs. false negatives
6. ✓ **Use cross-validation** to ensure results generalize

## Bonus Challenge (Optional)

If you finish early or want extra practice, try these extensions:

### Challenge 1: NearMiss Undersampling
Implement a pipeline using `NearMiss` undersampling (from `imblearn.under_sampling`). NearMiss intelligently selects majority class samples near the decision boundary instead of random sampling.

### Challenge 2: Hybrid Approach
Create a pipeline that combines both undersampling and oversampling:
- First undersample the majority class (e.g., to 3:1 ratio)
- Then use SMOTE to balance fully

### Challenge 3: Custom Class Weights
Instead of `class_weight='balanced'`, try custom weights:
```python
class_weight={0: 1, 1: 10}  # or different ratios
class_weight={0: 1, 1: 20}
```
Find the optimal weight ratio for this dataset.

### Challenge 4: Different Classifiers
Try the same techniques with different classifiers:
- Random Forest
- Gradient Boosting
- Support Vector Machine

Which classifier + imbalance handling combination works best?

## Summary

Congratulations! You've learned how to:
- ✓ Identify and understand class imbalance problems
- ✓ Build classification pipelines with scikit-learn and imblearn
- ✓ Implement multiple strategies for handling imbalance
- ✓ Evaluate models using appropriate metrics
- ✓ Compare approaches and select the best solution

These skills are essential for real-world machine learning, where imbalanced data is common in fraud detection, medical diagnosis, anomaly detection, and many other domains.

**Remember**: The "best" approach depends on your specific problem, dataset size, computational resources, and business requirements. Always experiment with multiple strategies and evaluate thoroughly!